In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import numpy as np
from time import sleep
import json

from URBasic import Joint6D
from URBasic import TCP6D
from URBasic import TCP6DDescriptor

import src.calibration as Calibration
import src.space as Space

SIMULATION = True
# Create a new ISCoin object
# UR3e1 IP (closest to window): 10.30.5.158
# UR3e2 IP: 10.30.5.159
robot_ip = "10.30.5.159"
robot_opened_gripper_size_mm = 40. # Max oppenning of gripper

In [4]:
file_path = "trapezoid_letters-trapezoid_letters-trace.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
print(data["generated_at"])

2026-02-26T14:37:43.905785


In [ ]:
p_world = np.array(data["calibration"])
if len(p_world) < 4:
    print("Missing point")

In [ ]:
from URBasic import ISCoin

iscoin = ISCoin(robot_ip, robot_opened_gripper_size_mm)

if not iscoin.isInitOk:
    print("Error robot not initialised")
    sys.exit(1)

iscoin.robot_control.reset_error()

In [ ]:
robot_arm = iscoin.robot_control
tcps = Calibration.collect_data(robot_arm, 20)
tcp_offset = Calibration.get_tcp_offset(tcps)

robot_arm.set_tcp(tcp_offset)
robot_arm.freedrive_mode()
tcp_ref = None
while not tcp_ref:
    i = input(f"Confirm reference position for TCP validation. y/n")
    if i.lower()[0] == "y":
        robot_arm.end_freedrive_mode()
        joint_ref = robot_arm.get_actual_joint_positions()

Calibration.validate_calibration(robot_arm, joint_ref)

In [ ]:
p_tcp = Space.collect_data(robot_arm, p_world)
world_to_tcp = Space.affine_transform_rot_3D(p_world, p_tcp)

In [ ]:
traces = data["traces"]
draw_motions = []
move_motions = []
above = [0,0,0.02,0,0,0]
for trace in traces:
    normal = trace["face"]
    p_world = trace["path"]
    motion = []
    
    # Drawing motion
    for i, p in enumerate(p_world):
        x,y,z,rx,ry,rz = world_to_tcp(p, normal)
        
        # Add entry point
        if i == 0:
            x_t,y_t,z_t,rx_t,ry_t,rz_t = Space.pose_trans([x,y,z,rx,ry,rz], above)
            t = TCP6D.createFromMetersRadians(x,y,z,rx,ry,rz)
            m = TCP6DDescriptor(t, a=5 ,v=1, t=0)
            motion.append(m)

        # Add drawing point
        t = TCP6D.createFromMetersRadians(x,y,z,rx,ry,rz)
        m = TCP6DDescriptor(t, a=5 ,v=1, t=0)
        motion.append(m)

        # Add exit point
        if i == len(p_world)-1:
            x_t,y_t,z_t,rx_t,ry_t,rz_t = Space.pose_trans([x,y,z,rx,ry,rz], above)
            t = TCP6D.createFromMetersRadians(x,y,z,rx,ry,rz)
            m = TCP6DDescriptor(t, a=5 ,v=1, t=0)
            motion.append(m)
    
    draw_motions.append(motion)
    move_motions.append((motion[0], motion[-1]))

In [ ]:
move_outside = []
for m in range(len(move_motions)):
    if m == 0:
        move_outside.append([TCP6DDescriptor(tcp_ref, a=5 ,v=1, t=0), move_motions[m][0]])
    elif m == len(move_motions):
        move_outside.append([move_motions[m][1], TCP6DDescriptor(tcp_ref, a=5 ,v=1, t=0)])
    else:
        move_outside.append([move_motions[m-1][1], move_motions[m][0]])

In [ ]:
for k in range(len(move_motions)):
    py.move(move_outside[k])
    robot_arm.movel_waypoints(draw_motions[k])

In [ ]:
traced = {
    "generated_at": "2026-02-26T09:32:30Z",
    "model": "trapezoid.obj",
    "texture": "trapezoid.png",
    "calibration": [
        [-30, 35.753, -19.358],
        [-30, 35.753, 19.358],
        [30, 35.753, 19.358],
        [60, 0, -40]
    ],
    "traces": [
        {
            "face": [0, 1, 0],
            "color": 0,
            "path": [
                [0, 0, 0],
                [0, 0.2, 0],
                [0.2, 0.2, 0]
            ]
        },
        {
            "face": [0, 0, 1],
            "color": 1,
            "path": [
                [0, 0, 0],
                [0, 0.2, 0],
                [0.2, 0.2, 0]
            ]
        }
    ]
}